In [1]:
import sys
sys.path.insert(0, "/home/go274/scratch_pi_skr2/go274/manual_MPRA_models/alphaMPRA")

import jax, jax.numpy as jnp, numpy as np
from alphagenome.models import dna_output
from alphagenome_research.model import dna_model
from alphagenome_ft import HeadConfig, HeadType, register_custom_head, load_checkpoint
from alphampra import EncoderMPRAHead

STAGE2 = "/home/go274/scratch_pi_skr2/go274/manual_MPRA_models/alphaMPRA/multihead_episomal_alphampra/checkpoints/episomal_alphampra_multihead/stage2"
BASE = "/home/go274/scratch_pi_skr2/go274/coda_data/claude_playground/alphagenome_weights/all_folds"
CELLS = ["K562", "HepG2", "SKNSH"]            # head order used in training
HEADS = [f"mpra_head_{c}" for c in CELLS]

# 1. register the three heads exactly as in training (same class and metadata)
for head in HEADS:
    register_custom_head(head, EncoderMPRAHead, HeadConfig(
        type=HeadType.GENOME_TRACKS, output_type=dna_output.OutputType.RNA_SEQ, num_tracks=1,
        metadata={"center_bp": 256, "pooling_type": "flatten", "nl_size": [512, 512], "do": 0.1, "activation": "relu"}))

# 2. load the folder
model = load_checkpoint(STAGE2, base_checkpoint_path=BASE, init_seq_len=600)

# 3. predict
strand_reindex = jax.device_put(model._metadata[dna_model.Organism.HOMO_SAPIENS].strand_reindexing,
                                model._device_context._device)

def predict(seqs):
    """list of 600bp strings -> (N, 3) predicted log2FC, columns K562, HepG2, SKNSH"""
    x = jnp.asarray(np.stack([model._one_hot_encoder.encode(s) for s in seqs]))   # (N, 600, 4)
    with model._device_context:
        out = model._predict(model._params, model._state, x, jnp.zeros(len(seqs), dtype=jnp.int32),
                             negative_strand_mask=jnp.zeros(len(seqs), dtype=bool),
                             strand_reindexing=strand_reindex, requested_outputs=tuple(dna_output.OutputType))
    return np.concatenate([np.asarray(out[h], dtype=np.float32).reshape(-1, 1) for h in HEADS], axis=1)


I0000 00:00:1790781002.148842 3301173 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790781016.107633 3301173 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
ERROR:jax._src.xla_bridge:Jax plugin configuration error: Exception when calling jax_plugins.xla_cuda12.initialize()
Traceback (most recent call last):
  File "/home/go274/.conda/envs/alphampra/lib/python3.11/site-packages/jax/_src/xla_bridge.py", line 508, in discover_pjrt_plugins
    plugin_module.initialize()
  File "/home/go274/.conda/envs/alphampra/lib/python3.11/site-packages/jax_plugins/xla_cuda12/__init__.py", line 370, in ini

Loading checkpoint from /nfs/roberts/scratch/pi_skr2/go274/manual_MPRA_models/alphaMPRA/multihead_episomal_alphampra/checkpoints/episomal_alphampra_multihead/stage2
  Custom heads: ['mpra_head_K562', 'mpra_head_HepG2', 'mpra_head_SKNSH']
  Model type: Minimal (encoder + heads only)
Loading pretrained AlphaGenome model...
  Using local checkpoint at: /home/go274/scratch_pi_skr2/go274/coda_data/claude_playground/alphagenome_weights/all_folds


/home/go274/.conda/envs/alphampra/lib/python3.11/site-packages/pyfaidx/__init__.py:596: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)
/home/go274/.conda/envs/alphampra/lib/python3.11/site-packages/pyfaidx/__init__.py:596: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)


✓ Pretrained model loaded
Initializing heads: ['mpra_head_K562', 'mpra_head_HepG2', 'mpra_head_SKNSH']
Initializing parameters... (seq_len=600)
✓ Head parameters initialized
Merging pretrained backbone with heads...
✓ Parameters merged
✓ Model created successfully
  Total parameters: 463,049,352
  Heads: ['mpra_head_K562', 'mpra_head_HepG2', 'mpra_head_SKNSH']
Loading minimal model from checkpoint (encoder + custom heads only)...
  Creating model structure with encoder-only forward (init_seq_len=600bp)...
Loading pretrained AlphaGenome model...
  Using local checkpoint at: /home/go274/scratch_pi_skr2/go274/coda_data/claude_playground/alphagenome_weights/all_folds


/home/go274/.conda/envs/alphampra/lib/python3.11/site-packages/pyfaidx/__init__.py:596: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)
/home/go274/.conda/envs/alphampra/lib/python3.11/site-packages/pyfaidx/__init__.py:596: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)


✓ Pretrained model loaded
Initializing heads: ['mpra_head_K562', 'mpra_head_HepG2', 'mpra_head_SKNSH']
Initializing parameters... (seq_len=600)
✓ Head parameters initialized
Merging pretrained backbone with heads...
✓ Parameters merged
✓ Model created successfully
  Total parameters: 463,049,352
  Heads: ['mpra_head_K562', 'mpra_head_HepG2', 'mpra_head_SKNSH']
✓ Minimal model loaded (encoder + custom heads, encoder-only forward)
✓ Checkpoint loaded successfully
  Total parameters: 463,049,352
